In [ ]:
%pip install sentence_transformers

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

from src.data import (
    load_declarations,
    load_regulations,
)

from src.preprocessing import (
    declaration_to_text,
    regulation_to_text,
)

from src.bm25 import BM25Retriever
from src.embeddings import EmbeddingRetriever
from src.retrieval import (
    reciprocal_rank_fusion,
)

c:\HSE\testNLP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Анализ данных

In [2]:
declarations = load_declarations(
    "./data/declarations (4).jsonl"
)

regulations = load_regulations(
    "./data/regulations (4).jsonl"
)
declarations.head()

,declaration_id,G011,G221,G31_1,G32,G34,G42,GD0,GD00,ND,desc_extention,has_acceptance_docs
0,6091ab306df3425437ab3113,ИМ,,СИСТЕМА ХРАНЕНИЯ ДАННЫХ DATA STORAGE SYSTEM МО...,140,,,10,None,None,:,0
1,609190fc6df3425437aef1af,ИМ,,"АППАРАТУРА ПЕРЕДАЮЩАЯ, ВКЛЮЧАЮЩАЯ В СВОЙ СОСТА...",4,,,10,None,None,:,0
2,60926ff56df3425437dac7c5,ИМ,,"НАСОСЫ МОЛЕКУЛЯРНЫЕ (ВАКУУМНЫЕ), ПРОМЫШЛЕННЫЕ,...",86,,,10,None,None,:,0
3,60941fc06df34254375e2f72,ИМ,,"НАСОСЫ МОЛЕКУЛЯРНЫЕ (ВАКУУМНЫЕ) ПРОМЫШЛЕННЫЕ, ...",90,,,10,None,None,:,0
4,60933b696df34254372e4337,ИМ,,5.2 ИМУЩЕСТВО ПО ПЕРЕЧНЮ №1XXXXX7: ПОЗ. ПО ЛИЦ...,29,,,10,None,None,:,0


In [3]:
regulations.head()

,regulation_id,decree_number,npa,source
0,NPA0001,1661,"Раздел 1, 6.1.4.1.4. зеркала, специально разра...",NaN
1,NPA0002,1661,"Раздел 1, 1.3.12.2. предварительно обогащенный...",NaN
2,NPA0003,1661,"Раздел 1, 8.1.2.1.3. системы, оборудование и к...",NaN
3,NPA0004,36,Угловые измерительные приборы с отклонением уг...,NaN
4,NPA0005,1082,"Дистилляционные или абсорбционные колонны, кот...",NaN


In [4]:
print("declaration shape:", declarations.shape)
print("regulation shape:", regulations.shape)

# G32
print("\nG32:")
print("  unique:", declarations["G32"].nunique())
print("  values:", sorted(declarations["G32"].unique())[:10], "...")

# source
print("\nsource:")
print("  non-null:", regulations["source"].notna().sum())
print("  unique non-null:", regulations["source"].dropna().unique())

# Пустые и константные колонки
empty_or_constant = [
    col for col in declarations.columns
    if declarations[col].isna().all()
    or declarations[col].nunique(dropna=False) == 1
]

print("\nПустые или константные колонки:", empty_or_constant)

# Удаляем поля, которые не несут информации для сопоставления
declarations = declarations.drop(columns=empty_or_constant + ["G32"])
regulations = regulations.drop(columns=["source"])

print("\nОставшиеся колонки деклараций:", declarations.columns.tolist())
print("Оставшиеся колонки НПА:", regulations.columns.tolist())

declaration shape: (151, 12)
regulation shape: (562, 4)

G32:
  unique: 151
  values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ...

source:
  non-null: 19
  unique non-null: ['annotation']

Пустые или константные колонки: ['G011', 'G221', 'G34', 'G42', 'GD0', 'GD00', 'ND', 'desc_extention', 'has_acceptance_docs']

Оставшиеся колонки деклараций: ['declaration_id', 'G31_1']
Оставшиеся колонки НПА: ['regulation_id', 'decree_number', 'npa']


В декларациях поле `G32` содержит 151 уникальное значение для 151 декларации — значения представляют собой числа от 1 до 151 и не повторяются. Поэтому оно не добавляет информации для сопоставления декларации с НПА и исключается из дальнейшего анализа.

Остальные поля деклараций, содержащие только пропуски или одно постоянное значение, также не несут информации для ранжирования и исключаются.

В данных НПА поле `source` заполнено только в 19 из 562 записей, причём единственное встречающееся значение — `annotation`. Поэтому оно также исключается как практически неинформативное для задачи сопоставления.

В результате для дальнейшего анализа используются текстовое описание товара `G31_1` в декларации и текст НПА `npa`. Поля `declaration_id` и `regulation_id` сохраняются как идентификаторы, а `decree_number` — как дополнительная метаинформация о НПА.


In [5]:
print(regulations["decree_number"].nunique())
print(regulations["decree_number"].value_counts().head(20))

7
decree_number
1661      350
202        64
36         58
1005       50
1083       20
1082       12
КЕЭК30      8
Name: count, dtype: int64


In [6]:
# Длины текстов

declarations["text_len"] = declarations["G31_1"].fillna("").str.len()
declarations["word_len"] = declarations["G31_1"].fillna("").str.split().str.len()

regulations["text_len"] = regulations["npa"].fillna("").str.len()
regulations["word_len"] = regulations["npa"].fillna("").str.split().str.len()

print("Длина описаний деклараций:")
print(declarations["text_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nКоличество слов в декларациях:")
print(declarations["word_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nДлина НПА:")
print(regulations["text_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nКоличество слов в НПА:")
print(regulations["word_len"].describe(percentiles=[.5, .75, .9, .95, .99]))


# Пропуски и дубликаты

print("\nПустые тексты:")
print("Декларации:", (declarations["G31_1"].fillna("").str.strip() == "").sum())
print("НПА:", (regulations["npa"].fillna("").str.strip() == "").sum())

print("\nДубликаты текстов:")
print("Декларации:", declarations["G31_1"].duplicated().sum())
print("НПА:", regulations["npa"].duplicated().sum())

print("\nДубликаты ID:")

print("declaration_id:" , declarations["declaration_id"].duplicated().sum())
print("regulation_id:" , regulations["regulation_id"].duplicated().sum())

Длина описаний деклараций:
count     151.000000
mean      431.695364
std       358.399832
min        48.000000
50%       311.000000
75%       555.500000
90%       906.000000
95%      1210.000000
99%      1634.500000
max      2131.000000
Name: text_len, dtype: float64

Количество слов в декларациях:
count    151.000000
mean      53.350993
std       45.788528
min        5.000000
50%       39.000000
75%       67.000000
90%      116.000000
95%      145.000000
99%      201.000000
max      280.000000
Name: word_len, dtype: float64

Длина НПА:
count     562.000000
mean      362.005338
std       334.592659
min        17.000000
50%       259.500000
75%       431.000000
90%       760.100000
95%       939.750000
99%      1549.110000
max      4000.000000
Name: text_len, dtype: float64

Количество слов в НПА:
count    562.000000
mean      45.912811
std       44.043220
min        2.000000
50%       32.000000
75%       53.000000
90%       97.900000
95%      128.000000
99%      198.290000
max      492

Тексты деклараций и НПА не содержат пропусков. Длина варьируется от коротких фрагментов до достаточно длинных описаний (до 280 слов для деклараций и до 492 слов для НПА). Тексты имеют уникальные ID, при этом встречаются отдельные одинаковые тексты, которые сохраняются как разные записи.

## Baseline

In [ ]:
declarations["text"] = declarations.apply(
    declaration_to_text,
    axis=1,
)

regulations["text"] = regulations.apply(
    regulation_to_text,
    axis=1,
)

In [ ]:
bm25 = BM25Retriever(
    regulations["text"].tolist()
)

In [13]:
for i in [0, 10, 20]:
    query = declarations.iloc[0]["text"]

    results = bm25.retrieve(query, top_k=5)

    print("=" * 80)
    print("DECLARATION:", declarations.iloc[i]["declaration_id"])
    print(query[:500])

    for rank, (idx, score) in enumerate(results, 1):
        print(f"\n{rank}. {regulations.iloc[idx]['regulation_id']} | score={score:.3f}")
        print(regulations.iloc[idx]["text"][:300])

DECLARATION: 6091ab306df3425437ab3113
система хранения данных data storage system мод.e03t (scv3020) память: 15 твердотельных накопителей по 1.92 tb, sas, 12 гбит/с+ 15 т вердотельных накопителей по 2.4 tb, sas, 12 гбит/с плата ввода-вывода, fc 16 гбит/с, 4 порта, pci-e, полновысотная

1. NPA0190 | score=23.470
раздел 1, 3.1.2.1.6. устройства записи цифровых данных, удовлетворяющие всем следующим условиям: а) обладающие устойчивой пропускной способностью диска или твердотельной памяти более 6,4 гбит/с; и б) использующие процессор, выполняющий анализ параметров радиочастотного сигнала одновременно с его зап

2. NPA0013 | score=18.850
раздел 1, 3.1.2.7. электронные сборки, модули или оборудование, предназначенные для выполнения всего следующего: а) аналого-цифровых преобразований, имеющих любую из следующих характеристик: разрешающую способность 8 бит или более, но менее 10 бит с частотой выборки более 1,3 млрд. выборок в секунду

3. NPA0364 | score=12.401
раздел 1, 1.3.2.1.1. алюминиды 

In [ ]:
embedding_retriever = EmbeddingRetriever(
    model_path="./models/embedding",
    device="cuda",
    batch_size=32,
    use_e5_prefix=True,
)

FileNotFoundError: Path ../models/embedding not found

In [ ]:

embedding_retriever.fit(
    regulations["text"].tolist()
)

embedding_results = (
    embedding_retriever.retrieve(
        query,
        top_k=10,
    )
)

for idx, score in embedding_results:
    print(
        regulations.iloc[idx]["regulation_id"],
        score,
    )

## Создание validation датасета

In [ ]:
from src.llm_validation import make_llm_prompt

prompt = make_llm_prompt(
    declaration_id=declaration_id,
    declaration_text=declaration_text,
    candidates=candidates,
)

print(prompt)

In [ ]:
answer = """{
  "declaration_id": "37",
  "results": [...]
}"""
from src.llm_validation import validate_llm_answer

answer = validate_llm_answer(
    answer,
    candidate_ids
)

In [ ]:
from src.llm_validation import save_llm_answer

save_llm_answer(
    answer,
    "experiments/llm_answers.jsonl"
)

In [ ]:
bm25_10 = {
    idx for idx, _ in bm25.retrieve(
        query,
        top_k=10,
    )
}

embedding_10 = {
    idx for idx, _ in embedding_retriever.retrieve(
        query,
        top_k=10,
    )
}

print("BM25:", len(bm25_10))
print("Embedding:", len(embedding_10))
print("Intersection:", len(
    bm25_10 & embedding_10
))
print("Only BM25:", len(
    bm25_10 - embedding_10
))
print("Only embedding:", len(
    embedding_10 - bm25_10
))

In [ ]:
bm25_results = bm25.retrieve(
    query,
    top_k=50,
)

embedding_results = (
    embedding_retriever.retrieve(
        query,
        top_k=50,
    )
)

fused = reciprocal_rank_fusion(
    [
        bm25_results,
        embedding_results,
    ],
    k=60,
    top_k=20,
)

for idx, score in fused:
    print(
        regulations.iloc[idx]["regulation_id"],
        score,
        regulations.iloc[idx]["text"][:200],
    )

In [ ]:
from src.reranker import Reranker

reranker = Reranker(
    model_path="../models/reranker",
    device="cuda",
)

In [ ]:
candidate_indices = [
    idx for idx, _ in fused
]

candidates = [
    (
        idx,
        regulations.iloc[idx]["text"],
    )
    for idx in candidate_indices
]

reranked = reranker.rerank(
    query,
    candidates,
    top_k=10,
)

for idx, score in reranked:
    print(
        regulations.iloc[idx]["regulation_id"],
        score,
    )
    print(
        regulations.iloc[idx]["text"]
    )
    print()